Quiz 44 — Gym Member Segmentation  [SOLVED]
Difficulty: Hard

In [ ]:
import numpy as np

np.random.seed(70)
n = 50
ages            = np.random.randint(18, 65, n)
visits          = np.random.randint(1, 30, n)
membership_type = np.random.choice(["Basic","Silver","Gold"], n)
monthly_spend   = np.random.uniform(20, 150, n)

# ── 1. Tier segmentation ─────────────────────────────────────────────────────
tiers = np.select(
    [visits < 5, visits <= 14, visits <= 22],
    ["Inactive", "Casual",    "Active"],
    default="Elite"
)

# ── 2. Stats per tier ────────────────────────────────────────────────────────
print("Activity Tier Analysis:")
for tier in ["Inactive","Casual","Active","Elite"]:
    mask = tiers == tier
    if mask.sum() == 0:
        continue
    print(f"  {tier:8s}: count={mask.sum():2d}  "
          f"mean_spend=£{monthly_spend[mask].mean():.2f}  "
          f"mean_age={ages[mask].mean():.1f}")

# ── 3. Top-10 spenders ───────────────────────────────────────────────────────
top10_idx  = np.argsort(monthly_spend)[-10:][::-1]
top_types  = membership_type[top10_idx]
print("\nTop-10 spenders membership types:", top_types)
from collections import Counter
print("  Distribution:", dict(Counter(top_types.tolist())))

# ── 4. At-risk Gold members ──────────────────────────────────────────────────
gold_mask   = membership_type == "Gold"
gold_visits = visits[gold_mask]
gold_median = np.median(gold_visits)
at_risk     = (gold_mask) & (visits < gold_median)
print(f"\nGold median visits: {gold_median:.1f}")
print(f"At-risk Gold (below median): {at_risk.sum()}")

# ── 5. Top spenders — what % are Gold? ───────────────────────────────────────
p80        = np.percentile(monthly_spend, 80)
top_spend  = monthly_spend > p80
gold_pct   = (membership_type[top_spend] == "Gold").mean() * 100
print(f"\n80th pct spend threshold: £{p80:.2f}")
print(f"Members above P80 who are Gold: {gold_pct:.1f}%")

# ── WHY ───────────────────────────────────────────────────────────────────────
# np.select handles multi-level tiering cleanly — equivalent to SQL CASE WHEN.
# Compound boolean masks (gold_mask) & (visits < gold_median) narrow scope
# precisely without loops.